In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
from fasthtml.common import A, Button as FhButton, I, Span
from fh_matui.foundations import normalize_tokens, dedupe_preserve_order, stringify, listify, VEnum

---
description: drop in replacement for monster ui with material design for fasthtml
output-file: core.html
title: core

---

In [ ]:
#| export

HEADER_URLS = {
    "beercss_css": "https://cdn.jsdelivr.net/npm/beercss@3.13.1/dist/cdn/beer.min.css",
    "beercss_js": "https://cdn.jsdelivr.net/npm/beercss@3.13.1/dist/cdn/beer.min.js",
    "mdc_js": "https://cdn.jsdelivr.net/npm/material-dynamic-colors@1.1.2/dist/cdn/material-dynamic-colors.min.js",
}

########################################


# All BeerCSS color names
COLOR_NAMES = ['amber', 'blue', 'blue_grey', 'brown', 'cyan', 'deep_orange', 'deep_purple', 
               'green', 'grey', 'indigo', 'light_blue', 'light_green', 'lime', 'orange', 
               'pink', 'purple', 'red', 'teal', 'yellow']

# All shared helpers organized by category
SIZES = ['tiny', 'small', 'medium', 'large', 'extra', 'wrap', 'no_wrap', 'max']
WIDTH_HEIGHT = ['auto_width', 'small_width', 'medium_width', 'large_width',
                'auto_height', 'small_height', 'medium_height', 'large_height']
ELEVATES = ['elevate', 'no_elevate', 'small_elevate', 'medium_elevate', 'large_elevate']
DIRECTIONS = ['horizontal', 'vertical']
FORMS = ['border', 'no_border', 'circle', 'square', 'none', 'fill', 'extend', 'tabbed', 
         'round', 'no_round', 'small_round', 'medium_round', 'large_round',
         'left_round', 'right_round', 'top_round', 'bottom_round']
MARGINS = ['margin', 'no_margin', 'auto_margin', 'tiny_margin', 'small_margin', 'medium_margin', 'large_margin',
           'left_margin', 'right_margin', 'top_margin', 'bottom_margin', 'horizontal_margin', 'vertical_margin']
PADDINGS = ['padding', 'no_padding', 'tiny_padding', 'small_padding', 'medium_padding', 'large_padding',
            'left_padding', 'right_padding', 'top_padding', 'bottom_padding', 'horizontal_padding', 'vertical_padding']
POSITIONS = ['left', 'right', 'center', 'top', 'bottom', 'middle', 'front', 'back']
RESPONSIVE = ['responsive', 's', 'm', 'l']
ALIGNMENTS = ['left_align', 'right_align', 'center_align', 'top_align', 'bottom_align', 'middle_align']
BLURS = ['blur', 'small_blur', 'medium_blur', 'large_blur']
OPACITIES = ['opacity', 'no_opacity', 'small_opacity', 'medium_opacity', 'large_opacity']
SHADOWS = ['shadow', 'left_shadow', 'right_shadow', 'top_shadow', 'bottom_shadow']
SPACES = ['space', 'no_space', 'small_space', 'medium_space', 'large_space']
RIPPLES = ['ripple', 'slow_ripple', 'fast_ripple']
SCROLLS = ['scroll', 'no_scroll']
WAVES = ['wave', 'no_wave']
ZOOMS = ['zoom', 'tiny_zoom', 'small_zoom', 'medium_zoom', 'large_zoom', 'extra_zoom']
THEME_HELPERS = ['light', 'dark', 'primary', 'secondary', 'tertiary', 'transparent',
                 'primary_text', 'primary_border', 'primary_container',
                 'secondary_text', 'secondary_border', 'secondary_container',
                 'tertiary_text', 'tertiary_border', 'tertiary_container',
                 'error', 'error_text', 'error_border', 'error_container',
                 'background', 'surface', 'surface_variant', 'inverse_surface',
                 'inverse_primary', 'inverse_primary_text', 'inverse_primary_border',
                 'black', 'black_text', 'black_border',
                 'white', 'white_text', 'white_border',
                 'transparent_text', 'transparent_border']
TYPOGRAPHY = ['italic', 'bold', 'underline', 'overline', 'upper', 'lower', 'capitalize', 
              'link', 'small_text', 'medium_text', 'large_text']
TRIGGERS = ['active']

# Generate color variants (color1-10, color, color-border, color-text)
COLOR_HELPERS = []
for color in COLOR_NAMES:
    for i in range(1, 11):
        COLOR_HELPERS.append(f'{color}{i}')
    COLOR_HELPERS.extend([color, f'{color}_border', f'{color}_text'])


# Combine all helpers
ALL_HELPERS = (SIZES + WIDTH_HEIGHT + ELEVATES + DIRECTIONS + FORMS + MARGINS + PADDINGS + 
               POSITIONS + RESPONSIVE + ALIGNMENTS + BLURS + OPACITIES + SHADOWS + SPACES + 
               RIPPLES + SCROLLS + WAVES + ZOOMS + THEME_HELPERS + TYPOGRAPHY + TRIGGERS + COLOR_HELPERS)


beer_hdrs = (
    Link(href=HEADER_URLS["beercss_css"], rel='stylesheet', type='text/css'),
    Script(src=HEADER_URLS["beercss_js"], type='module'),
    Script(src=HEADER_URLS["mdc_js"], type='module'),
   
)

In [ ]:
#| export

class _ThemeChain:
    def __init__(self, color=None):
        self._color = color
    
    def headers(self, title="App", mode="auto"):
        hdrs = list(beer_hdrs)
        if self._color:
            # Beer CSS theming - map color names to HEX values for ui("theme") function
            color_map = {
                'amber': '#ffc107', 'blue': '#2196f3', 'red': '#f44336', 'green': '#4caf50',
                'purple': '#9c27b0', 'orange': '#ff9800', 'pink': '#e91e63', 'cyan': '#00bcd4',
                'teal': '#009688', 'indigo': '#3f51b5', 'lime': '#cddc39', 'grey': '#9e9e9e',
                'brown': '#795548', 'yellow': '#ffeb3b', 'light_blue': '#03a9f4',
                'light_green': '#8bc34a', 'deep_orange': '#ff5722', 'deep_purple': '#673ab7',
                'blue_grey': '#607d8b'
            }
            
            hex_color = color_map.get(self._color, '#673ab7')  # Default to deep_purple
            
            theme_script = Script(f'''
                // Initialize theme using correct Beer CSS approach with HEX colors
                window.addEventListener("load", () => {{
                    function initTheme() {{
                        if (typeof ui !== 'undefined') {{
                            try {{
                                ui("theme", "{hex_color}");
                                console.log("✓ Beer CSS theme applied: {self._color} -> {hex_color}");
                                console.log("✓ CSS custom properties set for Material Design");
                            }} catch (e) {{
                                console.error("Beer CSS theme error:", e.message);
                            }}
                        }} else {{
                            console.warn("Beer CSS ui() function not available");
                        }}
                        
                        // Set mode using ui() function
                        if (typeof ui !== 'undefined' && "{mode}" !== "auto") {{
                            try {{
                                ui("mode", "{mode}");
                                console.log("✓ Beer CSS mode set:", "{mode}");
                            }} catch (e) {{
                                console.error("Beer CSS mode error:", e.message);
                            }}
                        }}
                    }}
                    
                    // Multiple attempts to ensure ui() is loaded
                    setTimeout(initTheme, 50);
                    setTimeout(initTheme, 200);
                    setTimeout(initTheme, 500);
                }});
                
                // Simplified setTheme function - accepts hex colors directly
                window.setTheme = function(hexColor) {{
                    if (typeof ui !== 'undefined') {{
                        try {{
                            ui("theme", hexColor);
                            console.log("✓ Theme changed to:", hexColor);
                        }} catch (e) {{
                            console.error("Theme change error:", e.message);
                        }}
                    }}
                }};
                
                window.toggleMode = function() {{
                    if (typeof ui !== 'undefined') {{
                        try {{
                            const currentMode = ui("mode");
                            const newMode = currentMode === "dark" ? "light" : "dark";
                            ui("mode", newMode);
                            console.log("✓ Mode toggled:", currentMode, "->", newMode);
                            return newMode;
                        }} catch (e) {{
                            console.error("Mode toggle error:", e.message);
                        }}
                    }}
                }};
                
                window.setMode = function(mode) {{
                    if (typeof ui !== 'undefined') {{
                        try {{
                            ui("mode", mode);
                            console.log("✓ Mode set to:", mode);
                        }} catch (e) {{
                            console.error("Mode set error:", e.message);
                        }}
                    }}
                }};
                
                // Custom toggleNav function for navigation rails
                // Beer CSS ui() function only handles 'active' class, not 'max' needed for nav rails
                window.toggleNav = function(selector) {{
                    const nav = document.querySelector(selector);
                    if (nav) {{
                        nav.classList.toggle('max');
                        console.log("✓ Navigation toggled:", selector, nav.classList.contains('max') ? 'expanded' : 'collapsed');
                    }} else {{
                        console.error("toggleNav: Element not found:", selector);
                    }}
                }};
                
            ''')
            hdrs.append(theme_script)
        hdrs.append(Title(title))
        return tuple(hdrs)

In [ ]:
#| export

# Create Theme namespace with color properties
class _ThemeNamespace:
    @property
    def amber(self): return _ThemeChain("amber") 
    
    @property
    def blue(self): return _ThemeChain("blue")
    
    @property
    def red(self): return _ThemeChain("red")
    
    # Additional colors mapped to BeerCSS equivalents
    @property
    def slate(self): return _ThemeChain("grey")  # BeerCSS grey for slate
    
    @property
    def stone(self): return _ThemeChain("brown")  # BeerCSS brown for stone
    
    @property
    def gray(self): return _ThemeChain("grey")  # BeerCSS grey (direct match)
    
    @property
    def grey(self): return _ThemeChain("grey")  # BeerCSS grey (alternative spelling)
    
    @property
    def neutral(self): return _ThemeChain("grey")  # BeerCSS grey for neutral
    
    @property
    def rose(self): return _ThemeChain("pink")  # BeerCSS pink for rose
    
    @property
    def orange(self): return _ThemeChain("orange")  # BeerCSS orange (direct match)
    
    @property
    def green(self): return _ThemeChain("green")  # BeerCSS green (direct match)
    
    @property
    def yellow(self): return _ThemeChain("yellow")  # BeerCSS yellow (direct match)
    
    @property
    def violet(self): return _ThemeChain("purple")  # BeerCSS purple for violet
    
    @property
    def purple(self): return _ThemeChain("purple")  # BeerCSS purple (direct match)
    
    @property
    def zinc(self): return _ThemeChain("blue_grey")  # BeerCSS blue_grey for zinc
    
    # Additional BeerCSS native colors for completeness
    @property
    def cyan(self): return _ThemeChain("cyan")
    
    @property
    def teal(self): return _ThemeChain("teal")
    
    @property
    def indigo(self): return _ThemeChain("indigo")
    
    @property
    def pink(self): return _ThemeChain("pink")
    
    @property
    def lime(self): return _ThemeChain("lime")
    
    @property
    def light_blue(self): return _ThemeChain("light_blue")
    
    @property
    def light_green(self): return _ThemeChain("light_green")
    
    @property
    def deep_orange(self): return _ThemeChain("deep_orange")
    
    @property
    def deep_purple(self): return _ThemeChain("deep_purple")
    
    @property
    def blue_grey(self): return _ThemeChain("blue_grey")
    
    @property
    def brown(self): return _ThemeChain("brown")

Theme = _ThemeNamespace()

########################################

# Base Beer CSS helper chain class
class BeerCssChain:
    """Base class for chaining Beer CSS helper classes together"""
    def __init__(self, tokens=None):
        self._tokens = list(tokens or [])
    
    def __iter__(self):
        return iter(self._tokens)
    
    def __str__(self):
        return " ".join(self._tokens)
    
    def __repr__(self):
        return f"BeerCssChain({self._tokens})"

# Generate properties for all Beer CSS helpers
for name in ALL_HELPERS:
    css_name = name.replace('_', '-')
    setattr(BeerCssChain, name, property(lambda self, css=css_name: type(self)(self._tokens + [css])))

########################################